# Tennis stroke classifier: training

Trains a small 1D CNN on `dataset.npz` (made by **Build training set** in `label_app.py`, or `python strokes.py`).
Each example is the near player's skeleton over ±1 s around a hit: 31 time steps × 53 numbers.

**Colab:** Runtime → Run all. When asked, upload `data/dataset.npz`. A GPU isn't needed; this trains in about a minute on CPU.
**Locally:** run it from the repo root with the project venv; it finds `data/dataset.npz` by itself.

Output: `stroke_model.pt` (weights + labels + normalisation). Put it in the repo root.

In [ ]:
import os, json, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

SEED = 0
BLOCK_S = 30.0      # hits within the same 30 s of video always land on the same side of a split
EPOCHS = 80
BATCH = 32
LR = 1e-3
torch.manual_seed(SEED); np.random.seed(SEED)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("torch", torch.__version__, "| Colab" if IN_COLAB else "| local")

## 1 · Load the dataset

In [ ]:
path = next((p for p in ("dataset.npz", "data/dataset.npz") if os.path.exists(p)), None)
if path is None and IN_COLAB:
    from google.colab import files
    path = next(iter(files.upload()))          # pick data/dataset.npz from your Mac
d = np.load(path)
X, y_all, ALL_LABELS = d["X"].astype(np.float32), d["y"], [str(l) for l in d["labels"]]

# keep only classes that actually have examples (practice rallying has no serves/volleys yet)
present = [i for i in range(len(ALL_LABELS)) if (y_all == i).sum() > 0]
LABELS = [ALL_LABELS[i] for i in present]
remap = {old: new for new, old in enumerate(present)}
y = np.array([remap[v] for v in y_all])
counts = np.bincount(y, minlength=len(LABELS))
print(f"{len(X)} hits, input shape {X.shape[1:]}")
for l, c in zip(LABELS, counts):
    print(f"  {l:10s} {c:4d}  ({100 * c / len(y):.0f}%)")
print(f"Always guessing '{LABELS[counts.argmax()]}' would already be {100 * counts.max() / len(y):.0f}% right; the model has to beat that.")

## 2 · How the model is tested
Neighbouring hits in a rally look alike, so a random split would let the model "cheat" by seeing near-copies of the test hits.
Instead, hits are grouped into 30-second blocks of video and whole blocks go to either training or testing.
With only a couple of hundred hits, a single 80/20 split is noisy, so we do it 5 times (5-fold cross-validation):
every hit is tested exactly once, by a model that never saw its block.

In [ ]:
groups = np.array([f"{v}:{int(t // BLOCK_S)}" for v, t in zip(d["video"], d["time_s"])])
print(len(set(groups)), "blocks of", BLOCK_S, "s")

def make_model(n_feat, n_cls):
    return nn.Sequential(
        nn.Conv1d(n_feat, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
        nn.Conv1d(64, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
        nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64, n_cls))

def augment(xb):
    # small random time shift (hit timing from audio is ±1 frame) + a little noise
    shift = np.random.randint(-2, 3)
    xb = torch.roll(xb, shift, dims=2)
    return xb + 0.05 * torch.randn_like(xb)

def fit(Xtr, ytr, verbose=False):
    mean = Xtr.reshape(-1, Xtr.shape[2]).mean(0)
    std = Xtr.reshape(-1, Xtr.shape[2]).std(0) + 1e-6
    xt = torch.tensor((Xtr - mean) / std).transpose(1, 2)          # (N, features, time)
    yt = torch.tensor(ytr)
    w = torch.tensor(len(ytr) / (len(LABELS) * np.maximum(np.bincount(ytr, minlength=len(LABELS)), 1)),
                     dtype=torch.float32)                          # rare classes count more
    model = make_model(Xtr.shape[2], len(LABELS))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2: continue
            opt.zero_grad()
            loss = loss_fn(model(augment(xt[b])), yt[b])
            loss.backward(); opt.step()
        if verbose and (ep + 1) % 20 == 0:
            print(f"  epoch {ep + 1}: loss {loss.item():.3f}")
    model.eval()
    return model, mean, std

def predict(model, mean, std, Xs):
    with torch.no_grad():
        return model(torch.tensor((Xs - mean) / std).transpose(1, 2)).softmax(1).numpy()

## 3 · Cross-validate

In [ ]:
n_splits = min(5, int(counts.min()))           # every class needs at least one example per fold
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
pred = np.zeros(len(y), int)
for k, (tr, te) in enumerate(cv.split(X, y, groups)):
    model, mean, std = fit(X[tr], y[tr])
    pred[te] = predict(model, mean, std, X[te]).argmax(1)
    print(f"fold {k + 1}/{n_splits}: {len(te)} test hits, {100 * (pred[te] == y[te]).mean():.0f}% right")

print(f"\nOverall: {100 * (pred == y).mean():.1f}% right (always-majority baseline {100 * counts.max() / len(y):.0f}%)\n")
print(classification_report(y, pred, target_names=LABELS, digits=2, zero_division=0))
cm = confusion_matrix(y, pred, labels=range(len(LABELS)))
w = max(len(l) for l in LABELS)
print("Confusion matrix (rows = your label, columns = model's guess):")
print(" " * (w + 2) + "  ".join(f"{l[:8]:>8s}" for l in LABELS))
for l, row in zip(LABELS, cm):
    print(f"{l:>{w}s}  " + "  ".join(f"{v:8d}" for v in row))

**Reading this:** *recall* = of the hits you labelled X, how many the model also called X.
*precision* = of the hits the model called X, how many really were X.
Classes with few examples (e.g. 16 `not_a_hit`) have very noisy scores; more labels fix that faster than any model change.

## 4 · Train the final model on everything and save it

In [ ]:
model, mean, std = fit(X, y, verbose=True)
out = "stroke_model.pt"
torch.save({
    "state_dict": model.state_dict(),
    "labels": LABELS,
    # plain tensors/lists only, so torch.load(..., weights_only=True) works on any torch version
    "mean": torch.tensor(mean), "std": torch.tensor(std),
    "n_features": int(X.shape[2]), "grid": [float(g) for g in d["grid"]],
    "arch": "conv1d-64x2-v1",
    "cv_accuracy": float((pred == y).mean()),
    "n_train": int(len(y)),
}, out)
print(f"Saved {out} ({os.path.getsize(out) / 1024:.0f} KB), classes {LABELS}")
if IN_COLAB:
    from google.colab import files
    files.download(out)                        # then move it into the repo root